In [1]:
# %pip install --upgrade transformers --break-system-packages

In [2]:
# %pip install auto-round --break-system-packages

In [3]:
# %pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [4]:
# %pip install compressed-tensors --break-system-packages

In [5]:
import os
import shutil

import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor

W1011 14:02:36.116000 42660 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1011 14:02:36.148000 42660 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [6]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [7]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.14.0+cu132
CUDA Available: True
CUDA Version: 13.2
GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB


In [8]:
notebook_login()

In [9]:
MODEL_ID = "vllm-sr/d3-mini"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [10]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 32 files:   0%|                                | 0/32 [00:00<?, ?it/s]Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)

Reconstructing (incomplete total...):   0%|       |  0.00B / 11.4kB            
Reconstructing (incomplete total...):   1%|       | 11.4kB / 2.24MB            

Fetching 32 files:   3%|▊                       | 1/32 [00:00<00:10,  2.84it/s]
Reconstructing (incomplete total...):   1%|       | 11.4kB / 2.25MB            
Reconstructing (incomplete total...):   1%|       | 22.9kB / 2

In [11]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [12]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
         

In [13]:
TUNING_CONFIG = {
    "group_size": 32,
    "sym": True,
    "iters": 1200,                  # High accuracy (Production grade)
    "nsamples": 512,                # More calibration data
    "batch_size": 4,
    "seqlen": 4096,
    "low_gpu_mem_usage": False,     # Keep on GPU for speed
    "enable_torch_compile": True,   # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
}

In [14]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-10-11 14:03:00 WARNING autoround.py L489: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-10-11 14:03:00 WARNING autoround.py L489: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-10-11 14:03:00 WARNING autoround.py L489: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [15]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round", inplace=True
)

2026-10-11 14:03:01 INFO base.py L1365: `torch.compile` is enabled
2026-10-11 14:03:01 INFO orchestrator.py L594: start to cache block inputs
2026-10-11 14:03:01 INFO mllm.py L86: Using MLLM template: qwen3_5
2026-10-11 14:03:01 INFO mllm.py L125: Multimodal model with non-MLLM calibration dataset 'NeelNanda/pile-10k' and quant_nontext_module=False: using the standard text dataloader (vision/audio towers are not being quantized, so text-only calibration through the full-model forward is sufficient).
2026-10-11 14:03:01 INFO calib_dataset.py L1229: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-10-11 14:03:14 INFO device.py L1560: 'peak_ram': 20.49GB, 'peak_vram': 8.56GB
2026-10-11 14:03:14 INFO orchestrator.py L626: caching done
Quantizing model.language_model.layers.0:   0%|          | 0/32 [00:02<?, ?it/s]quantized 8/8 layers in the block, loss iter 0: 2.468e-05 -> iter 1126: 1.988e-06
2026-10-11 14:05:35 INFO device.py L1560: 'peak_ram': 27.21GB, 'pe

(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 1024)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-23): 24 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=1024, out_features=3072, bias=True)
             (proj): Linear(in_features=1024, out_features=1024, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
             (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
             (act_fn): 

In [16]:
# %%
# Files that must accompany the quantized backbone for downstream inference
auxiliary_files = [
    # Custom scripts & modeling code
    "modeling_d3.py",
    "pipeline_d3.py",
    "d3_engine.py",
    "d3_fast.py",
    "d3_format.py",
    "d3_kernels.py",
    "d3_runtime.py",
    "d3_server.py",
    # Dedicated readout head weights & configs
    "readout.safetensors",
    "decision_config.json",
    "MODEL_MANIFEST.json",
    # Templates & preprocessing configs
    "chat_template.jinja",
    "video_preprocessor_config.json",
    "requirements.txt",
]

# Note: Uses w4g32 to match your TUNING_CONFIG group_size: 32
export_subdirs = [
    os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32"),
]

for export_dir in export_subdirs:
    if os.path.exists(export_dir):
        print(f"\n--> Injecting d3-edge custom runtime assets into: {export_dir}")
        for filename in auxiliary_files:
            src = os.path.join(LOCAL_PATH, filename)
            dst = os.path.join(export_dir, filename)
            if os.path.exists(src):
                shutil.copy2(src, dst)
                print(f"    Copied: {filename}")
            else:
                print(f"    Warning: Missing {src}")
    else:
        print(f"\n--> Target export directory not found: {export_dir}")


--> Injecting d3-edge custom runtime assets into: ./AutoRound/local_model-w4g32
    Copied: modeling_d3.py
    Copied: pipeline_d3.py
    Copied: d3_engine.py
    Copied: d3_fast.py
    Copied: d3_format.py
    Copied: d3_kernels.py
    Copied: d3_runtime.py
    Copied: d3_server.py
    Copied: readout.safetensors
    Copied: decision_config.json
    Copied: MODEL_MANIFEST.json
    Copied: chat_template.jinja
    Copied: video_preprocessor_config.json
    Copied: requirements.txt


In [17]:
def push_to_hub(local_dir, repo_name, token):
    if not os.path.exists(local_dir):
        print(f"[Hub] Directory not found, skipping: {local_dir}")
        return

    full_repo_id = f"{HF_USER}/{repo_name}"
    try:
        api = HfApi()
        print(f"[Hub] Preparing repo: {full_repo_id}...")
        create_repo(full_repo_id, repo_type="model", private=False, token=token, exist_ok=True)
        
        print(f"[Hub] Uploading folder from {local_dir}...")
        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            token=token
        )
        print(f"[Hub] Upload complete: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] Error uploading {full_repo_id}: {e}")


In [18]:

base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()


In [19]:

if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32"), 
        f"{base_name}-W4A16-AutoRound", 
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload.")

[Hub] Preparing repo: Vishva007/d3-mini-W4A16-AutoRound...
[Hub] Uploading folder from ./AutoRound/local_model-w4g32...
[Hub] Upload complete: https://huggingface.co/Vishva007/d3-mini-W4A16-AutoRound
